# Day 35 — PD 프로젝트 보고서 쓰기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-5/day-35.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-5/day-35.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 2 — 보고서에 들어갈 숫자를 한 번에 다시 만들기

보고서의 숫자를 여러 노트북에서 손으로 옮기면 반드시 어긋납니다. 핵심 숫자를 한 셀에서 다시 계산해 문장 형태로 출력하는 코드입니다. 가상 데이터로 실행한 결과이며, 내 데이터로 바꾸면 내 보고서의 문장 초안이 됩니다.

In [ ]:
import numpy as np
import pandas as pd

FEATURES = ["stride_time_cv", "accel_rms", "tremor_band_power",
            "jerk_mean", "step_regularity", "arm_swing_asym"]

def make_demo_table(seed=42, n_participants=50, n_repeats=6):
    """연습용 가상 feature table. 실제 환자 데이터가 아니다."""
    rng = np.random.default_rng(seed)
    pd_shift = np.array([0.6, -0.5, 0.6, 0.2, -0.4, 0.3])   # PD군 평균이 이동하는 정도
    rows = []
    for i in range(n_participants):
        group = "PD" if i < n_participants // 2 else "HC"
        age = rng.normal(68 if group == "PD" else 62, 7)
        sex = rng.choice(["F", "M"])
        person = rng.normal(0, 1.0, size=len(FEATURES))      # 그 사람 고유의 '움직임 지문'
        for rep in range(1, n_repeats + 1):
            values = person + rng.normal(0, 0.25, size=len(FEATURES))  # 측정마다 생기는 흔들림
            if group == "PD":
                values = values + pd_shift
            values[0] += 0.06 * (age - 65)                   # stride_time_cv는 나이와도 관련
            rows.append({"participant_id": f"P{i + 1:03d}", "group": group,
                         "age": round(age, 1), "sex": sex, "repeat": rep,
                         **dict(zip(FEATURES, values.round(3)))})
    return pd.DataFrame(rows)

df = make_demo_table()

# 🔁 실제 데이터로 바꿀 지점 ---------------------------------------------
# 4주차(Day 26)에 만든 feature table이 있으면 위의 df = make_demo_table() 대신:
# df = pd.read_csv("data/physionet_pd/derived/pd_features_records.csv")   # 내 파일 경로로
# FEATURES = ["내_feature_1", "내_feature_2"]                # 내 feature 열 이름으로
# 그리고 아래 두 이름을 내 표의 열 이름에 맞춘다.
ID_COL = "participant_id"   # 참가자 ID 열
GROUP_COL = "group"         # 진단군 열 (이 예시에서는 "PD" / "HC")
# -------------------------------------------------------------------------

import sklearn
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MAIN = "stride_time_cv"      # 🔁 Day 28에 정한 주요 outcome으로

# --- 표본 ---
person = df.groupby([ID_COL, GROUP_COL], as_index=False)[["age"] + FEATURES].mean()
n_pd = (person[GROUP_COL] == "PD").sum()
n_hc = (person[GROUP_COL] == "HC").sum()

# --- 주 분석: 효과크기와 bootstrap 신뢰구간 (Day 30) ---
def cohens_d(a, b):
    pooled = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(pooled)

a = person.loc[person[GROUP_COL] == "PD", MAIN].to_numpy()
b = person.loc[person[GROUP_COL] == "HC", MAIN].to_numpy()
rng = np.random.default_rng(0)
boots = [cohens_d(rng.choice(a, len(a)), rng.choice(b, len(b))) for _ in range(5000)]
d_lo, d_hi = np.percentile(boots, [2.5, 97.5])

# --- 교란 점검: 나이 보정 전·후 (Day 31) ---
term = f"C({GROUP_COL}, Treatment('HC'))"
crude = smf.ols(f"{MAIN} ~ {term}", data=person).fit().params[f"{term}[T.PD]"]
adjusted = smf.ols(f"{MAIN} ~ {term} + age", data=person).fit().params[f"{term}[T.PD]"]

# --- 보조 분석: 참가자 단위 교차검증 (Day 33–34) ---
X, y, groups = df[FEATURES], (df[GROUP_COL] == "PD").astype(int), df[ID_COL]
cv = GroupKFold(n_splits=5)
proba = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()),
                          X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
fold_auc = [roc_auc_score(y.iloc[test], proba[test]) for _, test in cv.split(X, y, groups)]

print(f"[표본] 참가자 {len(person)}명 (PD {n_pd}, HC {n_hc}), 측정 {len(df)}개")
print(f"[주 분석] {MAIN}: Cohen's d = {cohens_d(a, b):.2f} (95% CI {d_lo:.2f}–{d_hi:.2f}), PD − HC")
print(f"[교란 점검] 군 차이: 보정 전 {crude:.2f} → 나이 보정 후 {adjusted:.2f}")
print(f"[보조 분석] 참가자 단위 5-fold CV: ROC-AUC = {roc_auc_score(y, proba):.2f} "
      f"(fold 범위 {min(fold_auc):.2f}–{max(fold_auc):.2f}), accuracy = {accuracy_score(y, proba >= 0.5):.2f}")
print(f"[환경] pandas {pd.__version__}, numpy {np.__version__}, scikit-learn {sklearn.__version__}")

### 예시 3 — 과장 표현 점검기

보고서 초안에서 조심해야 할 표현을 찾아 주는 작은 도구입니다. 표현이 걸렸다고 무조건 틀린 것은 아닙니다("진단이 확정된 참가자"처럼 정당한 쓰임도 있습니다). 걸린 줄을 사람이 다시 읽어 보라는 표시입니다.

In [ ]:
from pathlib import Path

WATCH_WORDS = {
    "진단 암시": ["진단한다", "진단할 수", "진단 정확도", "선별", "조기 발견", "진단 보조"],
    "인과 암시": ["때문에", "원인", "유발", "증가시킨다", "감소시킨다", "증명"],
    "범위 넘침": ["임상적으로 유용", "일반적으로", "환자들은", "예측할 수 있다"],
    "종 간 직접 연결": ["마우스", "OSD-536", "우주비행"],
}

def check_overclaims(text):
    hits = []
    for line_no, line in enumerate(text.splitlines(), start=1):
        for kind, words in WATCH_WORDS.items():
            for w in words:
                if w in line:
                    hits.append((line_no, kind, w, line.strip()))
    return hits

report_path = Path("reports/pd_motor_phenotype_report.md")   # 🔁 내 보고서 경로
if report_path.exists():
    draft = report_path.read_text(encoding="utf-8")
else:   # 보고서가 아직 없으면 연습용 초안으로 실행
    draft = """이 모델은 파킨슨병을 73% 정확도로 진단한다.
PD 때문에 보행 주기 변동성이 증가한다는 것을 증명했다.
이 데이터셋에서 참가자 단위 교차검증으로 평가했을 때 baseline은 두 군의 측정을 ROC-AUC 0.83으로 구분했다.
이 결과는 마우스 뇌의 신경염증 결과와 일치하므로 스마트워치는 임상적으로 유용하다."""

for line_no, kind, word, line in check_overclaims(draft):
    print(f"{line_no}행 [{kind}] '{word}': {line}")
print("점검한 줄 수:", len(draft.splitlines()))